# Talk to Your Dataset — a Natural-Language Query Agent for FiftyOne

### Powered by Poolside **Laguna S 2.1** (open-weight coding agent) + **FiftyOne**

---

Type a question about your computer-vision dataset in plain English. A coding agent — Poolside's open-weight **Laguna S 2.1** — writes the corresponding FiftyOne query, runs it, **reads any error it hits, and rewrites the query until it works**. The result opens live in the FiftyOne App.

**Why this model?** Laguna S 2.1 is a 118B-parameter (8B active) MoE model built for *agentic coding*: write code, run it, read the result, persist until done. It has **no vision capability** — and that's the point. It never looks at a pixel; it reasons over the dataset *schema* and the Python *errors* it gets back, the way an engineer would. FiftyOne (and, where real perception is needed, a separate vision model) handles everything visual.

**Three parts:**
1. **Part I** — the core NL → FiftyOne view-query agent with self-correction.
2. **Part II** — the agent becomes a *router* across three tools: view queries, FiftyOne Brain (uniqueness / near-duplicates), and a vision model for pixel questions.
3. **Part III** — the whole thing packaged as a FiftyOne App **plugin panel**.

Everything here is self-contained and reproducible: a public model endpoint, a dataset that ships with FiftyOne, and no hard-coded paths.

## Prerequisites

1. **Python 3.9+ in a virtual environment** with FiftyOne installed:
   ```bash
   python -m venv .venv
   source .venv/bin/activate        # Windows: .venv\Scripts\activate
   pip install fiftyone openai
   ```
2. **Run Jupyter from that same environment** so this notebook's kernel is the one with FiftyOne. If you use multiple environments, register this one as a named kernel:
   ```bash
   python -m ipykernel install --user --name fiftyone-demo --display-name "Python (fiftyone-demo)"
   ```
   then select "Python (fiftyone-demo)" as the kernel. The first setup cell prints the active interpreter so you can confirm.
3. **An OpenRouter API key** (free tier works): create one at [openrouter.ai/keys](https://openrouter.ai/keys). Set it as an environment variable *before* launching Jupyter so both this notebook and the Part III App panel can read it:
   ```bash
   export OPENROUTER_API_KEY="sk-or-..."   # Windows: set OPENROUTER_API_KEY=sk-or-...
   ```

## 0 · Setup

Confirm the environment and install the OpenAI SDK (used purely as an HTTP client for OpenRouter's OpenAI-compatible endpoint).

In [ ]:
import sys
print('interpreter:', sys.executable)   # confirm this is your FiftyOne venv
!{sys.executable} -m pip install -q "openai>=1.40"

import fiftyone as fo
print('fiftyone   :', fo.__version__)
print('\n✅ environment ready')

### API access to Laguna S 2.1

We reach the model through **OpenRouter's free endpoint** (`poolside/laguna-s-2.1:free`), which speaks the OpenAI chat-completions protocol.

The key is read from the `OPENROUTER_API_KEY` environment variable. If it isn't set, the cell prompts for it and stores it in this kernel's environment so both the notebook and any App launched from this kernel can use it.

> **Local / offline option:** the weights are open (OpenMDW-1.1 on Hugging Face), so you can self-host with Ollama / vLLM / SGLang and point `BASE_URL` at your own server.

In [ ]:
import os, getpass

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key (sk-or-...): ')

# Free endpoint. Swap to 'poolside/laguna-s-2.1' for the paid 1M-context endpoint,
# or point BASE_URL at a local server if you self-host the open weights.
MODEL_ID = 'poolside/laguna-s-2.1:free'
BASE_URL = 'https://openrouter.ai/api/v1'

from openai import OpenAI
client = OpenAI(base_url=BASE_URL, api_key=os.environ['OPENROUTER_API_KEY'])
print('✅ Laguna client ready:', MODEL_ID)

> **Rotating your key?** If you regenerate the OpenRouter key, re-run the cell above so the `client` object picks up the new value — a `401 User not found` means the client is holding a stale key.

## 1 · Load a public dataset

We use FiftyOne's built-in **`quickstart`** dataset: ~200 images with ground-truth and predicted detections plus per-detection confidence. It ships with the library — nothing to download, no credentials. Everything below works the same on `coco-2017`, `bdd100k`, or your own dataset; the agent adapts to whatever schema it's given.

In [ ]:
import fiftyone.zoo as foz

DATASET_NAME = 'laguna-nlq-demo'

if DATASET_NAME in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME)   # fresh each run -> deterministic

source = foz.load_zoo_dataset('quickstart')
dataset = source.clone(DATASET_NAME)
dataset.persistent = True
print(dataset)

## 2 · Give the agent a map of the dataset

The agent can't see images, so we hand it the one thing it can reason about: a compact description of the dataset **schema** plus the object classes present. This is the agent's entire world.

In [ ]:
def describe_schema(ds):
    """Concise, LLM-friendly description of the dataset schema."""
    lines = [f"Dataset '{ds.name}': {ds.count()} samples.", '', 'Sample fields:']
    for name, field in ds.get_field_schema().items():
        ftype = type(field).__name__
        doc = getattr(field, 'document_type', None)
        detail = f' -> {doc.__name__}' if doc else ''
        lines.append(f'  - {name}: {ftype}{detail}')
    try:
        classes = ds.distinct('ground_truth.detections.label')
        lines += ['', 'Classes in ground_truth.detections.label:', '  ' + ', '.join(sorted(classes))]
    except Exception:
        pass
    return '\n'.join(lines)

SCHEMA_DESC = describe_schema(dataset)
print(SCHEMA_DESC)

## 3 · The agent loop (Part I)

The agent is given the schema and a plain-English request, and returns one line of Python that builds a FiftyOne view called `result`. We execute it in a sandbox; if it raises, we feed the traceback back and ask for a fix. This generate → run → read-error → repair cycle is exactly what Laguna was trained for.

The system prompt includes worked idioms so common requests land on the first attempt — the self-correction loop still fires for genuinely novel queries.

In [ ]:
import re, traceback
from fiftyone import ViewField as F

SYSTEM_PROMPT = f'''You are a FiftyOne query engineer. Translate the user's plain-English
request into a SINGLE Python expression that builds a filtered/sorted FiftyOne view.

You are given `dataset` (a fiftyone.Dataset) and `F` (fiftyone.ViewField, imported).
Use these plus standard view stages (match, filter_labels, sort_by, limit, etc.).

Rules:
- Assign your view to a variable named `result`. Example: result = dataset.match(...)
- Return ONLY a fenced python code block. No prose outside the block.
- Do NOT load images, print, or call .save(). Just build `result`.
- Counting ALL objects in a Detections field: F(\"field.detections\").length().
- Filtering objects within a sample: .filter_labels(\"field\", <F expr>).
- ViewField has NO .count_values() or .count() method. To count objects of a SPECIFIC
  class, filter the list first then take its length (see WORKED EXAMPLES).
- Bounding boxes are [x, y, width, height] normalized to [0, 1], so relative area is
  width * height (accessed as element 2 and 3 of the bounding_box list).

WORKED EXAMPLES (copy these patterns):
# More than 5 people:
result = dataset.match(F(\"ground_truth.detections\").filter(F(\"label\") == \"person\").length() > 5)
# Contains at least one dog:
result = dataset.match(F(\"ground_truth.detections\").filter(F(\"label\") == \"dog\").length() > 0)
# Has a car but no person:
result = dataset.match((F(\"ground_truth.detections\").filter(F(\"label\") == \"car\").length() > 0) & (F(\"ground_truth.detections\").filter(F(\"label\") == \"person\").length() == 0))
# Sort by number of objects, most first:
result = dataset.sort_by(F(\"ground_truth.detections\").length(), reverse=True)
# Top 10 by distinct classes:
result = dataset.sort_by(F(\"ground_truth.detections.label\").unique().length(), reverse=True).limit(10)
# Low-confidence predictions (objects, not whole samples):
result = dataset.filter_labels(\"predictions\", F(\"confidence\") < 0.2)
# Images with a small object (bbox area < 5% of image):
result = dataset.match(F(\"ground_truth.detections\").filter(F(\"bounding_box\")[2] * F(\"bounding_box\")[3] < 0.05).length() > 0)

DATASET SCHEMA:
{SCHEMA_DESC}
'''

def extract_code(text):
    m = re.search(r'```(?:python)?\s*(.*?)```', text, re.DOTALL)
    return (m.group(1) if m else text).strip()

def run_query(pycode):
    ns = {'dataset': dataset, 'fo': fo, 'F': F}
    exec(pycode, ns)
    if 'result' not in ns:
        raise ValueError('Code did not define a variable named `result`.')
    view = ns['result']
    _ = len(view)   # force evaluation so schema/label errors surface now
    return view

In [ ]:
def ask_dataset(question, max_repairs=3, verbose=True):
    """NL question -> validated FiftyOne view, with self-correction on errors."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': question},
    ]
    for attempt in range(1, max_repairs + 2):
        resp = client.chat.completions.create(
            model=MODEL_ID, messages=messages, temperature=0.1, max_tokens=1200)
        reply = resp.choices[0].message.content
        pycode = extract_code(reply)
        if verbose:
            print(f'── attempt {attempt} ── generated query:')
            print(pycode, '\n')
        try:
            view = run_query(pycode)
            if verbose:
                print(f'✅ success — {len(view)} samples matched\n')
            return view, pycode
        except Exception:
            tb = traceback.format_exc(limit=2)
            if verbose:
                print('⚠️  error — feeding it back to the agent:')
                print(tb)
            messages.append({'role': 'assistant', 'content': reply})
            messages.append({'role': 'user', 'content':
                f'That code raised this error. Fix it and return corrected python only:\n\n{tb}'})
    raise RuntimeError('Agent could not produce a working query within the retry budget.')

## 4 · Ask a question and see it in the App

The showpiece query for schema expressiveness — it combines an **object count**, a specific **class**, and a **bounding-box size** condition in one request. Tedious to write by hand; one sentence for the agent. (If the bbox-area math misses on the first attempt, you'll see the self-correction loop fix it.)

In [ ]:
QUESTION = ('Show me images with more than 5 people where at least one person is small '
            '- a bounding box under 5% of the image area.')
view, query_code = ask_dataset(QUESTION)

In [ ]:
session = fo.launch_app(view)
session

Each further call re-runs the agent and swaps the App's view. A few more to try:

In [ ]:
# Try any of these (or write your own):
#   'Show me images that contain both a dog and a cat.'
#   'Show me predicted detections with confidence below 0.2.'
#   'Find images with a very large object - a box covering more than half the image.'
view, query_code = ask_dataset('Show me images that contain both a dog and a cat.')
session.view = view

---

# Part II · Giving the agent more than one tool

View queries cover anything expressible over the schema. Two request types fall outside that: **similarity / uniqueness** (needs embeddings = FiftyOne Brain) and **pixel-level** questions (needs to actually look = a vision model). So we let Laguna **choose a tool**:

| Tool | When | Under the hood |
|------|------|----------------|
| `query_view` | schema-expressible filters/sorts | the Part I loop |
| `brain_op`   | similarity / uniqueness / duplicates | FiftyOne Brain |
| `vision_scan`| questions that need pixels | a vision model labels samples, then we filter |

Laguna stays the orchestrator and writes all the code; it only delegates the one thing it can't do — see.

## 5 · Tool 2 — FiftyOne Brain (uniqueness & near-duplicates)

`compute_uniqueness` adds a sortable `uniqueness` score; `compute_similarity` builds an embeddings index whose `find_duplicates()` surfaces near-dupes. Both download a default embedding model on first use (about a minute) and cache results via brain keys.

> **Tip for live demos:** run the uniqueness query once beforehand to pre-compute the embeddings, so the on-stage run is instant.

In [ ]:
import fiftyone.brain as fob

def ensure_uniqueness():
    if 'uniqueness' not in dataset.get_field_schema():
        print('⚙️  computing uniqueness (first run downloads a model)…')
        fob.compute_uniqueness(dataset)

def ensure_similarity(key='demo_sim'):
    if key not in dataset.list_brain_runs():
        print('⚙️  building similarity index (first run downloads a model)…')
        fob.compute_similarity(dataset, brain_key=key)
    return dataset.load_brain_results(key)

def brain_op(kind, k=25, threshold=None):
    """kind in {'unique','duplicates'} -> a FiftyOne view."""
    if kind == 'unique':
        ensure_uniqueness()
        return dataset.sort_by('uniqueness', reverse=True).limit(k)
    if kind == 'duplicates':
        index = ensure_similarity()
        index.find_duplicates(thresh=threshold)
        return index.duplicates_view()
    raise ValueError(f'unknown brain op: {kind!r}')

print('✅ brain_op ready  (kinds: unique, duplicates)')

## 6 · Tool 3 — a vision-language model for pixel questions

For *"find images taken outdoors"*, Laguna can't look, so it delegates: it picks a yes/no question and a field name, a **vision model** answers per image, we write the answer back as a boolean field, and Laguna's view-query skill filters on it.

This is built for the realities of free/shared endpoints, which rotate and rate-limit:
- **retries with backoff** on 429,
- **fails over to the next model** on a persistent 429 *or* a 404/delisted slug,
- runs requests **in parallel**,
- **registers the boolean field up front** (an unset FiftyOne field raises on read rather than returning None), and **saves each label as it's computed** so a failure never loses progress and a re-run resumes where it stopped.

Set `VISION_MODEL` to pin a model; otherwise the fallback list is tried free-first. OpenRouter's free VL catalog changes often — see [openrouter.ai/collections/vision-models](https://openrouter.ai/collections/vision-models) if all candidates fail.

In [ ]:
import base64, mimetypes, time, itertools, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

VISION_MODEL = None   # e.g. 'google/gemma-3-27b-it' (paid) to pin a specific model
VISION_FALLBACKS = [
    'google/gemma-4-31b-it:free',        # current free image+text pick
    'qwen/qwen2.5-vl-72b-instruct:free', # free Qwen VL
    'mistralai/mistral-small-3.1-24b-instruct:free',
    # paid below: not subject to the free shared pool, far more reliable (needs credits)
    'google/gemma-3-27b-it',
    'qwen/qwen3.7-flash',
]
MAX_WORKERS = 4
MAX_RETRIES = 4

class AllVisionModelsExhausted(RuntimeError):
    pass

def _encode_image(path):
    mime = mimetypes.guess_type(path)[0] or 'image/jpeg'
    with open(path, 'rb') as fh:
        return f'data:{mime};base64,' + base64.b64encode(fh.read()).decode()

def _is_rate_limit(e):
    return getattr(e, 'status_code', None) == 429 or '429' in str(e)

def _is_unavailable(e):
    code = getattr(e, 'status_code', None); s = str(e).lower()
    return code == 404 or '404' in s or 'unavailable' in s or 'no endpoints' in s

def _vlm_call(model, question, data_url):
    delay, last = 2.0, None
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=model, temperature=0, max_tokens=3,
                messages=[{'role': 'user', 'content': [
                    {'type': 'text', 'text': f'{question} Answer with exactly one word: yes or no.'},
                    {'type': 'image_url', 'image_url': {'url': data_url}},
                ]}])
            return (resp.choices[0].message.content or '').strip().lower()
        except Exception as e:
            last = e
            if _is_unavailable(e):
                raise                       # bad slug: fail fast, let failover skip it
            if _is_rate_limit(e) and attempt < MAX_RETRIES - 1:
                time.sleep(delay); delay *= 2
                continue
            raise
    raise last

def _scan_with_model(model, samples, question, field_name):
    print(f'👁️  scanning {len(samples)} images with {model} — "{question}"')
    lock, done = threading.Lock(), itertools.count(1)
    def work(sample):
        val = _vlm_call(model, question, _encode_image(sample.filepath)).startswith('y')
        with lock:
            sample[field_name] = val
            sample.save()                   # incremental: progress survives a crash
            n = next(done)
            if n % 10 == 0 or n == len(samples):
                print(f'    {n}/{len(samples)} labelled')
        return val
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        for fut in as_completed({pool.submit(work, s): s for s in samples}):
            fut.result()                    # re-raises so failover logic sees it

def vision_scan(question, field_name, max_samples=20):
    """VLM yes/no per image -> bool field; resilient to rate limits & bad slugs."""
    # Register the field first: an unset FiftyOne field raises on read, not None.
    if field_name not in dataset.get_field_schema():
        dataset.add_sample_field(field_name, fo.BooleanField)

    samples = list(dataset.limit(max_samples).iter_samples())
    if not samples:
        return dataset.limit(0)

    candidates = [VISION_MODEL] if VISION_MODEL else list(VISION_FALLBACKS)
    errors = []
    for model in candidates:
        pending = [s for s in samples if s.get_field(field_name) is None]  # resume support
        if not pending:
            break
        try:
            _scan_with_model(model, pending, question, field_name)
            print(f'✅ completed with {model}')
            break
        except Exception as e:
            labelled = sum(1 for s in samples if s.get_field(field_name) is not None)
            errors.append(f'  - {model}: {type(e).__name__}: {str(e)[:70]}')
            if _is_rate_limit(e) or _is_unavailable(e):
                why = 'rate-limited' if _is_rate_limit(e) else 'unavailable (bad/delisted slug)'
                print(f'⚠️  {model} {why} after {labelled}/{len(samples)} — failing over…')
                continue
            raise
    else:
        raise AllVisionModelsExhausted(
            'All vision models failed:\n' + '\n'.join(errors) +
            '\n\nAdd credits / your own provider key, lower MAX_WORKERS, or set VISION_MODEL. '
            'See https://openrouter.ai/collections/vision-models')

    return dataset.match(F(field_name) == True)

print('✅ vision_scan ready — retries, fails over across models, saves progress incrementally')

## 7 · The router — Laguna picks the tool

Given a request, Laguna returns a small JSON object naming the tool and its arguments. `query_view` reuses the self-correcting Part I loop; the others fill in parameters.

In [ ]:
import json as _json

ROUTER_PROMPT = '''You are the router for a computer-vision dataset assistant.
Choose exactly ONE tool for the user's request and return ONLY a JSON object.

Tools:
1. query_view  — filter/sort over dataset FIELDS (object counts, classes, confidence,
   bbox size, metadata). No pixels needed.  JSON: {"tool": "query_view"}
2. brain_op    — similarity/uniqueness/near-duplicates.
   JSON: {"tool": "brain_op", "kind": "unique"|"duplicates", "k": <int>}
3. vision_scan — ONLY when answering requires LOOKING AT THE IMAGE (indoor/outdoor,
   blur, lighting, colour, weather, scene mood).
   JSON: {"tool": "vision_scan", "question": "<yes/no question about one image>",
          "field_name": "<snake_case bool field>"}

Prefer query_view when the schema can answer it. Return only the JSON.'''

def route_and_run(question, verbose=True):
    resp = client.chat.completions.create(
        model=MODEL_ID, temperature=0, max_tokens=300,
        messages=[{'role': 'system', 'content': ROUTER_PROMPT},
                  {'role': 'user', 'content': question}])
    raw = resp.choices[0].message.content.strip()
    plan = _json.loads(raw[raw.find('{'): raw.rfind('}') + 1])
    if verbose:
        print('🧭  Laguna chose tool:', plan.get('tool'), '—', plan, '\n')
    tool = plan['tool']
    if tool == 'query_view':
        view, _ = ask_dataset(question, verbose=verbose)
        return view
    if tool == 'brain_op':
        return brain_op(plan['kind'], k=int(plan.get('k', 25)))
    if tool == 'vision_scan':
        return vision_scan(plan['question'], plan['field_name'])
    raise ValueError(f'router returned unknown tool: {tool!r}')

print('✅ router ready')

### The three finalized demo queries

One per tool — run them in order. Each shows a different capability.

In [ ]:
# TOOL 1 — view query (schema): counts + class + bbox size, no pixels.
session.view = route_and_run('Show me images with more than 5 people where at least one '
                             'person is small - a bounding box under 5% of the image area.')

In [ ]:
# TOOL 2 — FiftyOne Brain (embeddings): most-unique images.
# First run computes embeddings (~1 min); cached afterward.
session.view = route_and_run('Show me the 20 most unique images in the dataset.')

In [ ]:
# TOOL 3 — vision model (pixels): a question the schema CANNOT answer.
# Laguna delegates perception to a VLM, then filters on the result.
session.view = route_and_run('Find images taken outdoors.')

---

# Part III · Package it as an App plugin panel

A FiftyOne plugin is a directory with a `fiftyone.yml` manifest and a Python entrypoint that defines a panel. The panel adds a text box inside the App that calls the router and sets the current view.

**Two things that must be exactly right:**
- In `fiftyone.yml`, `panels:` is a **top-level** key listing panel **names** (matching `PanelConfig.name`). It is *not* nested under `fiftyone:`, and *not* a list of name/label dicts. A malformed manifest silently fails to register the panel.
- The panel runs in the **App server's process**, so it needs `openai` importable there and `OPENROUTER_API_KEY` in that environment. Launching the App from the same kernel (or shell) where the key is set handles this.

In [ ]:
import os

# Uses whatever plugins directory FiftyOne is configured to use (portable across machines).
plugins_dir = fo.config.plugins_dir or os.path.join(os.path.expanduser('~'), 'fiftyone', '__plugins__')
panel_dir = os.path.join(plugins_dir, 'talk-to-your-dataset')
os.makedirs(panel_dir, exist_ok=True)
print('installing plugin to:', panel_dir)

# Manifest: `panels:` is TOP-LEVEL and lists the panel NAME only.
with open(os.path.join(panel_dir, 'fiftyone.yml'), 'w') as fh:
    fh.write('''name: "@demo/talk-to-your-dataset"
type: plugin
version: 1.0.0
description: Natural-language dataset queries powered by Poolside Laguna S 2.1
fiftyone:
  version: ">=1.0"
panels:
  - talk_to_dataset
''')
print('✅ wrote fiftyone.yml')
print(open(os.path.join(panel_dir, 'fiftyone.yml')).read())

In [ ]:
# Panel entrypoint (full three-tool router: query_view + brain_op + vision_scan).
# - No on_change on the input (avoids focus loss while typing).
# - Guards None API responses.
# - vision_scan delegates pixel questions to a VLM: registers the bool field, retries
#   with backoff, fails over across models on 429/404, caps samples, reuses labels.
# - Reads OPENROUTER_API_KEY from the App server's environment.
panel_code = r'''import os, json, base64, mimetypes, time
import fiftyone as fo
import fiftyone.brain as fob
import fiftyone.operators as foo
import fiftyone.operators.types as types
from fiftyone import ViewField as F
from openai import OpenAI

MODEL_ID = "poolside/laguna-s-2.1:free"
BASE_URL = "https://openrouter.ai/api/v1"

VISION_FALLBACKS = [
    "google/gemma-4-31b-it:free",
    "qwen/qwen2.5-vl-72b-instruct:free",
    "mistralai/mistral-small-3.1-24b-instruct:free",
    "google/gemma-3-27b-it",   # paid, reliable
    "qwen/qwen3.7-flash",      # paid
]
VISION_MAX_SAMPLES = 20
VISION_MAX_RETRIES = 4

ROUTER_PROMPT = (
    "You are the router for a CV dataset assistant. Choose ONE tool and return ONLY "
    "JSON, no prose, no code fence.\n"
    "1. query_view -> {\"tool\":\"query_view\",\"code\":\"result = dataset...\"} using `dataset` "
    "and `F` (fiftyone.ViewField). For object counts/classes/confidence/metadata. "
    "Count a class like: result = dataset.match(F(\"ground_truth.detections\")"
    ".filter(F(\"label\")==\"bird\").length() > 0).\n"
    "2. brain_op -> {\"tool\":\"brain_op\",\"kind\":\"unique\",\"k\":20} for uniqueness.\n"
    "3. vision_scan -> {\"tool\":\"vision_scan\",\"question\":\"<yes/no question about ONE image>\","
    "\"field_name\":\"<snake_case bool field>\"} ONLY when answering requires LOOKING AT the "
    "image (indoor/outdoor, blur, lighting, weather, colour, scene). "
    "Example: outdoors -> {\"tool\":\"vision_scan\",\"question\":\"Is this photo taken outdoors?\","
    "\"field_name\":\"is_outdoor\"}.\n"
    "Prefer query_view when the schema can answer it. Return only JSON."
)

def _content(resp):
    try:
        c = resp.choices[0].message.content
    except (AttributeError, IndexError):
        return ""
    return c or ""

def _client():
    key = os.environ.get("OPENROUTER_API_KEY")
    if not key:
        raise RuntimeError("OPENROUTER_API_KEY is not set in the App server's environment.")
    return OpenAI(base_url=BASE_URL, api_key=key)

def _is_rate_limit(e):
    return getattr(e, "status_code", None) == 429 or "429" in str(e)

def _is_unavailable(e):
    code = getattr(e, "status_code", None); s = str(e).lower()
    return code == 404 or "404" in s or "unavailable" in s or "no endpoints" in s

def _encode_image(path):
    mime = mimetypes.guess_type(path)[0] or "image/jpeg"
    with open(path, "rb") as fh:
        return f"data:{mime};base64," + base64.b64encode(fh.read()).decode()

def _vlm_call(cli, model, question, data_url):
    delay, last = 2.0, None
    for attempt in range(VISION_MAX_RETRIES):
        try:
            r = cli.chat.completions.create(
                model=model, temperature=0, max_tokens=3,
                messages=[{"role": "user", "content": [
                    {"type": "text", "text": f"{question} Answer with exactly one word: yes or no."},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ]}])
            return _content(r).strip().lower()
        except Exception as e:
            last = e
            if _is_unavailable(e):
                raise
            if _is_rate_limit(e) and attempt < VISION_MAX_RETRIES - 1:
                time.sleep(delay); delay *= 2
                continue
            raise
    raise last

def _vision_scan(dataset, question, field_name):
    """VLM yes/no per image -> bool field, then return the matching view.
    Reuses existing labels if the field is already populated (no re-scan)."""
    if field_name not in dataset.get_field_schema():
        dataset.add_sample_field(field_name, fo.BooleanField)

    samples = list(dataset.limit(VISION_MAX_SAMPLES).iter_samples())
    pending = [s for s in samples if s.get_field(field_name) is None]

    if pending:
        cli = _client()
        model, errors = None, []
        for cand in VISION_FALLBACKS:
            try:
                _vlm_call(cli, cand, "Is this an image?", _encode_image(pending[0].filepath))
                model = cand
                break
            except Exception as e:
                errors.append(f"{cand}: {type(e).__name__}")
                continue
        if model is None:
            raise RuntimeError("No working vision model. Tried: " + "; ".join(errors))
        for s in pending:
            s[field_name] = _vlm_call(cli, model, question, _encode_image(s.filepath)).startswith("y")
            s.save()

    return dataset.match(F(field_name) == True)

def _run(dataset, question):
    cli = _client()
    r = cli.chat.completions.create(
        model=MODEL_ID, temperature=0, max_tokens=500,
        messages=[{"role": "system", "content": ROUTER_PROMPT},
                  {"role": "user", "content": question}])
    raw = _content(r).strip()
    if "{" not in raw or "}" not in raw:
        raise RuntimeError(f"Model returned no usable plan: {raw[:120]!r}")
    plan = json.loads(raw[raw.find("{"): raw.rfind("}") + 1])
    tool = plan.get("tool")

    if tool == "brain_op" and plan.get("kind") == "unique":
        if "uniqueness" not in dataset.get_field_schema():
            fob.compute_uniqueness(dataset)
        return dataset.sort_by("uniqueness", reverse=True).limit(int(plan.get("k", 20)))

    if tool == "vision_scan":
        return _vision_scan(dataset, plan["question"], plan["field_name"])

    ns = {"dataset": dataset, "F": F, "fo": fo}
    exec(plan["code"], ns)
    return ns["result"]

class TalkToDataset(foo.Panel):
    @property
    def config(self):
        return foo.PanelConfig(name="talk_to_dataset", label="Talk to Your Dataset")

    def on_load(self, ctx):
        ctx.panel.state.question = ""
        ctx.panel.state.status = "Ask a question about this dataset."

    def ask(self, ctx):
        q = (ctx.panel.state.question or "").strip()
        if not q:
            ctx.panel.state.status = "Type a question first."
            return
        ctx.panel.state.status = "Running... (visual questions scan images and may take a moment)"
        try:
            view = _run(ctx.dataset, q)
            ctx.ops.set_view(view=view)
            ctx.panel.state.status = f"OK - {len(view)} samples - {q}"
        except Exception as e:
            ctx.panel.state.status = f"Error - {type(e).__name__}: {e}"

    def render(self, ctx):
        panel = types.Object()
        # No on_change: avoids per-keystroke re-render that steals input focus.
        panel.str("question", label="Ask your dataset", view=types.FieldView())
        panel.btn("ask", label="Run query", on_click=self.ask, variant="contained")
        panel.md(f"**{ctx.panel.state.status}**")
        return types.Property(panel, view=types.GridView(gap=2, padding=2))

def register(p):
    p.register(TalkToDataset)
'''

with open(os.path.join(panel_dir, '__init__.py'), 'w') as fh:
    fh.write(panel_code)
print('✅ wrote __init__.py  (query_view + brain_op + vision_scan)')

In [ ]:
# Confirm registration. Names are printed (not raw objects) so it's readable.
import fiftyone.plugins as fop
names = [p.name for p in fop.list_plugins()]
print('Installed plugins:')
for n in names:
    print('  -', n)
if any('talk-to-your-dataset' in n for n in names):
    print('\n✅ Registered. RESTART the App (plugins are scanned at startup), then open')
    print('   the "+" tab → Talk to Your Dataset.')
else:
    print('\n⚠️  Not found. Check fiftyone.yml (panels: must be top-level) and that this')
    print('   kernel is the FiftyOne env. From a shell: `fiftyone plugins list`.')

### Launch the App with the plugin

Plugins are scanned at **server startup**, so a browser refresh won't surface a newly-written plugin — the App must be (re)launched. The panel needs `OPENROUTER_API_KEY` in this process's environment (set in cell 0), which is why we relaunch from this same kernel.

In [ ]:
session = fo.launch_app(dataset)   # re-launch so the new plugin is picked up
session

> **Running the App from a terminal instead?** Use the shell whose virtual environment has FiftyOne (check with `fiftyone --version`; use the `fiftyone` console script — note `python -m fiftyone` does not work). Then:
>
> ```bash
> export OPENROUTER_API_KEY="sk-or-..."
> fiftyone plugins list                 # expect @demo/talk-to-your-dataset
> fiftyone app launch laguna-nlq-demo
> ```
>
> If the panel doesn't appear, `fiftyone app debug laguna-nlq-demo` prints plugin load errors (e.g. a missing package in the App server's env) that the App UI otherwise hides.

---

## Recap

- **Part I** — Laguna turns English into a FiftyOne view query and repairs its own code from tracebacks. No pixels.
- **Part II** — one tool becomes a **router** across view queries, FiftyOne **Brain**, and a **vision model** for pixel questions. Laguna orchestrates and writes the code; it delegates only perception.
- **Part III** — the whole thing moves **inside the App** as a plugin panel.

A small, cheap, open-weight *coding* model makes a surprisingly strong dataset orchestrator: it reasons over schema and errors, picks tools, and writes the glue.

**Links:** [FiftyOne](https://voxel51.com/fiftyone/) · [FiftyOne docs](https://docs.voxel51.com/) · [Plugin development](https://docs.voxel51.com/plugins/developing_plugins.html) · [Poolside](https://poolside.ai/) · [OpenRouter](https://openrouter.ai/)

*Model: Poolside Laguna S 2.1 (OpenMDW-1.1) via OpenRouter. Vision: any OpenRouter multimodal model. Dataset: FiftyOne `quickstart`.*